# Generic THz spin-$1/2$ chain with one- and two-magnon manifolds

This example describes a finite periodic XXZ spin chain in a longitudinal static field, truncated at two spin deviations from its fully polarized state. It demonstrates:

- a magnetic Hamiltonian parametrized by $J_{xy}$, $J_z$, the Landé factor $g$, and the static field $B_0$;
- one shared polarized ground state for all one-magnon transitions;
- magnetic-dipole excitation by a spatially uniform THz field, selecting total momentum $q\simeq0$;
- transitions from the ground state to one magnon and from one to two magnons;
- ground-state-bleach, stimulated-emission, and bimagnon excited-state-absorption contributions; and
- numerical agreement and scaling differences between the dense-Liouville and sparse-sector backends.

Natural units are used internally, with $\hbar=1$. Energies are expressed in meV, magnetic fields in tesla, and $\mu_B=0.0578838$ meV/T. A physical delay in picoseconds is converted through $t_{\mathrm{internal}}=t_{\mathrm{ps}}/0.658212$, because $\hbar=0.658212$ meV ps.

In [ ]:
from itertools import combinations
from pathlib import Path
from time import perf_counter
import sys

import matplotlib.pyplot as plt
import numpy as np

# Locate the project root when the notebook is launched from examples/
# or from another directory inside the repository.
current_directory = Path.cwd().resolve()
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "projet_solver10.py").is_file():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise RuntimeError("Could not locate the directory containing projet_solver10.py.")

from projet_solver10 import (
    ExcitationSectorModel,
    FrequencyPathway,
    PropagationInterval,
    SpectroscopyPlotter,
    SpectroscopyProtocol,
    SpectroscopySolver,
    standard_nq_protocol,
)

## 1. Generic magnetic Hamiltonian and excitation manifolds

We use the number-conserving XXZ Hamiltonian

$$
H=J_{xy}\sum_j\left(S_j^xS_{j+1}^x+S_j^yS_{j+1}^y\right)
+J_z\sum_jS_j^zS_{j+1}^z
-g\mu_BB_0\sum_jS_j^z,
$$

with periodic boundary conditions. The reference state $|G\rangle=|\uparrow\cdots\uparrow\rangle$ is fully polarized along the static field. A magnon is one downward spin deviation, so $n_j=1/2-S_j^z$ and its creation operator is $m_j^\dagger=S_j^-$. Relative to the polarized-state energy,

$$
H=(g\mu_BB_0-J_z)\sum_jn_j
+\frac{J_{xy}}{2}\sum_j\left(m_j^\dagger m_{j+1}+m_{j+1}^\dagger m_j\right)
+J_z\sum_jn_jn_{j+1}.
$$

The Hilbert space is truncated after two magnons,

$$\mathcal H=\mathcal H_0\oplus\mathcal H_1\oplus\mathcal H_2,$$

with dimensions $1$, $L$, and $\binom{L}{2}$. The longitudinal exchange $J_z$ is also the nearest-neighbor magnon interaction: $J_z<0$ favors adjacent spin deviations and can produce a bimagnon bound state. The model is generic rather than fitted to one material; neglected terms include transverse static fields, Dzyaloshinskii--Moriya coupling, disorder, and interchain exchange.

In [ ]:
MU_B_MEV_PER_T = 0.0578838
HBAR_MEV_PS = 0.658212


def excitation_basis(n_sites, n_magnons):
    return tuple(combinations(range(n_sites), n_magnons))


def build_xxz_sector_hamiltonian(
    n_sites, n_magnons, J_xy, J_z, g_factor, static_field
):
    basis = excitation_basis(n_sites, n_magnons)
    state_index = {state: index for index, state in enumerate(basis)}
    hamiltonian = np.zeros((len(basis), len(basis)), dtype=complex)

    # Relative to the fully up-polarized reference state:
    # H = (g mu_B B0 - J_z) sum n_j
    #   + (J_xy / 2) sum (m_j^dagger m_{j+1} + h.c.)
    #   + J_z sum n_j n_{j+1}.
    one_magnon_cost = g_factor * MU_B_MEV_PER_T * static_field - J_z
    hopping = 0.5 * J_xy

    for column, state in enumerate(basis):
        occupied = set(state)
        adjacent_pairs = sum(
            (site + 1) % n_sites in occupied for site in occupied
        )
        hamiltonian[column, column] = (
            n_magnons * one_magnon_cost + J_z * adjacent_pairs
        )

        for site in state:
            for neighbor in ((site - 1) % n_sites, (site + 1) % n_sites):
                if neighbor in occupied:
                    continue
                target = tuple(sorted((occupied - {site}) | {neighbor}))
                row = state_index[target]
                hamiltonian[row, column] += hopping

    assert np.allclose(hamiltonian, hamiltonian.conj().T)
    return basis, hamiltonian


def build_magnon_creation_block(source_basis, target_basis, probe_profile):
    target_index = {state: index for index, state in enumerate(target_basis)}
    block = np.zeros((len(target_basis), len(source_basis)), dtype=complex)
    for column, state in enumerate(source_basis):
        occupied = set(state)
        for site, amplitude in enumerate(probe_profile):
            if site in occupied or amplitude == 0:
                continue
            target = tuple(sorted((*state, site)))
            block[target_index[target], column] += amplitude
    return block


def uniform_thz_probe_profile(n_sites):
    # A far-field THz wavelength is much longer than the lattice spacing.
    # The normalized profile therefore selects total momentum q approximately 0.
    return np.ones(n_sites, dtype=complex) / np.sqrt(n_sites)


def make_spin_chain_model(
    n_sites, J_xy, J_z, g_factor, static_field
):
    bases = {}
    hamiltonians = {}
    for n_magnons in (0, 1, 2):
        bases[n_magnons], hamiltonians[n_magnons] = (
            build_xxz_sector_hamiltonian(
                n_sites, n_magnons, J_xy, J_z, g_factor, static_field
            )
        )

    probe_profile = uniform_thz_probe_profile(n_sites)
    raising_blocks = {
        (1, 0): build_magnon_creation_block(
            bases[0], bases[1], probe_profile
        ),
        (2, 1): build_magnon_creation_block(
            bases[1], bases[2], probe_profile
        ),
    }

    model = ExcitationSectorModel(
        hamiltonians,
        raising_blocks,
        initial_sector=0,
    )
    return model, bases, hamiltonians, raising_blocks

## 2. One- and two-magnon spectra in the THz regime

In the one-magnon sector, the dispersion relative to the polarized ground state is

$$\omega(k)=g\mu_BB_0-J_z+J_{xy}\cos k.$$

The numerical values below describe a generic easy-axis ferromagnetic chain rather than a fit to one compound. The negative $J_{xy}$ places the bright $k=0$ magnon at the bottom of the band, while $J_z<0$ provides an attractive nearest-neighbor magnon interaction. The two-magnon transition energies enter the third-order signal through coherences between $\mathcal H_2$ and $\mathcal H_1$.

In [ ]:
n_sites = 6
J_xy = -0.30          # meV, transverse exchange
J_z = -0.55           # meV, longitudinal exchange and magnon attraction
g_factor = 2.10
static_field = 7.1773 # tesla; keeps the bright one-magnon energy near 1.1224 meV
eta = 0.03            # meV, phenomenological linewidth

model, bases, hamiltonians, raising_blocks = make_spin_chain_model(
    n_sites, J_xy, J_z, g_factor, static_field
)

one_magnon_cost = g_factor * MU_B_MEV_PER_T * static_field - J_z
one_magnon_energies = np.linalg.eigvalsh(hamiltonians[1])
two_magnon_energies = np.linalg.eigvalsh(hamiltonians[2])
k_points = 2.0 * np.pi * np.arange(n_sites) / n_sites
expected_one_magnon = one_magnon_cost + J_xy * np.cos(k_points)
bright_k0_energy = one_magnon_cost + J_xy

assert np.allclose(
    np.sort(one_magnon_energies), np.sort(expected_one_magnon), atol=1e-12
)
assert model.dimension(0) == 1
assert model.dimension(1) == n_sites
assert model.dimension(2) == n_sites * (n_sites - 1) // 2
assert model.initial_condition().sector == 0
assert np.allclose(
    np.abs(raising_blocks[(1, 0)][:, 0]),
    np.ones(n_sites) / np.sqrt(n_sites),
)

print("Generic XXZ parameters")
print(f"  J_xy = {J_xy:.3f} meV")
print(f"  J_z  = {J_z:.3f} meV")
print(f"  g     = {g_factor:.3f}")
print(f"  B_0   = {static_field:.4f} T")
print(f"  eta   = {eta:.3f} meV, corresponding to T2 ~ {HBAR_MEV_PS / eta:.2f} ps")
print("Sector dimensions:", {sector: model.dimension(sector) for sector in model.sectors()})
print(f"Total Hilbert dimension: {sum(model.dimension(s) for s in model.sectors())}")
print(
    f"One-magnon band: {one_magnon_energies.min():.4f} to "
    f"{one_magnon_energies.max():.4f} meV"
)
print(f"Uniform-THz bright k=0 energy: {bright_k0_energy:.4f} meV")
print(
    f"Two-magnon energies: {two_magnon_energies.min():.4f} to "
    f"{two_magnon_energies.max():.4f} meV"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
axes[0].plot(k_points, expected_one_magnon, "o-")
axes[0].plot(0.0, bright_k0_energy, "*", markersize=14, label="uniform-THz bright state")
axes[0].set_xlabel(r"Momentum $k$")
axes[0].set_ylabel("One-magnon energy (meV)")
axes[0].set_title("Generic XXZ one-magnon dispersion")
axes[0].legend()

axes[1].plot(
    np.arange(two_magnon_energies.size),
    np.sort(two_magnon_energies),
    "o",
    markersize=4,
)
axes[1].set_xlabel("Two-magnon eigenstate index")
axes[1].set_ylabel("Two-magnon energy (meV)")
axes[1].set_title(rf"Two-magnon manifold, $J_z={J_z:.2f}$ meV")

## 3. Excitation-sector model and backend agreement

`ExcitationSectorModel` diagonalizes each excitation manifold separately and transforms every rectangular transition block as

$$\widetilde M_{ts}=U_t^\dagger M_{ts}U_s.$$

The transverse magnetic field of the THz pulse couples through $M^x\propto\sum_j(S_j^++S_j^-)$. Starting from the up-polarized state, the excitation-raising part is $\sum_jS_j^-$: it increases the magnon number even though it lowers the microscopic spin projection. In the solver, its blocks are $M_{10}:\mathcal H_0\rightarrow\mathcal H_1$ and $M_{21}:\mathcal H_1\rightarrow\mathcal H_2$; their adjoints are generated automatically. Both numerical backends receive exactly the same model.

In [ ]:
sparse_solver = SpectroscopySolver(
    backend="sparse_sector", eta=eta, krylov_tolerance=1e-11
)
sparse_solver.feed_model(model)

dense_solver = SpectroscopySolver(
    backend="dense_liouville", eta=eta, cache_resolvents=False
)
dense_solver.feed_model(model)

assert sparse_solver.summary()["total_dimension"] == dense_solver.summary()["total_dimension"]
assert model.transition_decomposition() == "explicit_sector"

print("Sparse backend:", sparse_solver.summary()["backend"])
print("Dense backend:", dense_solver.summary()["backend"])
print("Sectors:", sparse_solver.summary()["sector_dimensions"])
print("Transition decomposition:", model.transition_decomposition())

## 4. Linear THz response and the $q\simeq0$ selection rule

A single ket-side excitation creates a coherence $|1\mathrm{mag}\rangle\langle G|$. Because a far-field THz wavelength is much larger than the lattice spacing, every site sees approximately the same magnetic field. The uniform operator therefore conserves crystal momentum and selects the $k=0$ one-magnon eigenstate of this ideal periodic chain. The other one-magnon eigenstates remain in the Hamiltonian but are dark in the linear spectrum. Dense and sparse calculations are evaluated on the same meV grid and must agree point by point.

In [ ]:
linear_pathway = FrequencyPathway(
    name="linear",
    interactions=("Ku",),
    component="linear",
)
linear_protocol = SpectroscopyProtocol(
    intervals=(PropagationInterval("omega", "frequency", coherence_order=1),),
    name="linear_frequency",
)

omega = np.linspace(0.90, 1.90, 101)  # meV


def calculate_linear_response(solver):
    return np.asarray(
        [
            solver.calc_pathway(
                linear_pathway, linear_protocol, {"omega": frequency}
            ).value
            for frequency in omega
        ]
    )


linear_sparse = calculate_linear_response(sparse_solver)
linear_dense = calculate_linear_response(dense_solver)
linear_backend_residual = np.max(np.abs(linear_sparse - linear_dense))
linear_peak_energy = omega[np.argmax(np.abs(linear_sparse))]

assert np.all(np.isfinite(linear_sparse))
assert np.allclose(linear_sparse, linear_dense, rtol=1e-8, atol=1e-9)
assert abs(linear_peak_energy - bright_k0_energy) < 2.0 * (omega[1] - omega[0])
print(f"Uniform-probe peak: {linear_peak_energy:.4f} meV")
print(f"Expected k=0 energy: {bright_k0_energy:.4f} meV")
print(f"Dense--sparse linear residual: {linear_backend_residual:.3e}")

In [ ]:
fig, axes = plt.subplots(
    2, 1, figsize=(7.2, 5.7), sharex=True, constrained_layout=True
)
axes[0].plot(omega, np.abs(linear_dense), label="dense", linewidth=2.2)
axes[0].plot(omega, np.abs(linear_sparse), "--", label="sparse")
axes[0].axvline(bright_k0_energy, color="0.45", linewidth=1.0, label=r"bright $k=0$")
axes[0].set_ylabel(r"$|M^{(1)}(\omega)|$")
axes[0].set_title(r"Uniform-THz one-magnon response ($q\simeq0$)")
axes[0].legend()
axes[1].semilogy(omega, np.maximum(np.abs(linear_sparse - linear_dense), 1e-18))
axes[1].set_xlabel(r"Energy $\omega$ (meV)")
axes[1].set_ylabel("Absolute difference")

## 5. Third-order rephasing response and bimagnon ESA

The ground-state-bleach and stimulated-emission pathways are

$$R_{\mathrm{GSB}}=(B_u,B_d,K_u),\qquad R_{\mathrm{SE}}=(B_u,K_u,B_d).$$

The two-magnon manifold adds the excited-state-absorption pathway

$$R_{\mathrm{ESA}}=(B_u,K_u,K_u).$$

All three pathways have coherence history $q=(-1,0,+1)$. In the ESA pathway, the final ket-side excitation maps $\mathcal H_1$ into $\mathcal H_2$, leaving a $|2\mathrm{mag}\rangle\langle1\mathrm{mag}|$ emission coherence. The uniform THz operator restricts this sequence to states compatible with zero total photon momentum. The attractive $J_z$ shifts the correlated two-magnon transitions away from the independent-magnon value.

In [ ]:
pathway_se = FrequencyPathway(
    name="SE",
    interactions=("Bu", "Ku", "Bd"),
    component="rephasing",
)
pathway_gsb = FrequencyPathway(
    name="GSB",
    interactions=("Bu", "Bd", "Ku"),
    component="rephasing",
)
pathway_esa = FrequencyPathway(
    name="ESA",
    interactions=("Bu", "Ku", "Ku"),
    component="rephasing",
)

protocol_1q = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)

omega_1q = np.linspace(-1.45, -0.85, 21)  # meV
omega_emit = np.linspace(0.55, 1.55, 25)  # meV
result_rephasing = sparse_solver.generate_spectrum(
    protocol_1q,
    axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
    fixed_coordinates={"t2": 0.0},
    pathways=(pathway_gsb, pathway_se, pathway_esa),
)

gsb = result_rephasing.pathways["GSB"]
se = result_rephasing.pathways["SE"]
esa = result_rephasing.pathways["ESA"]
rephasing = result_rephasing.components["rephasing"]
reconstruction_residual = np.max(np.abs(rephasing - (gsb + se + esa)))
reference_amplitude = max(np.max(np.abs(gsb)), np.max(np.abs(se)))

assert np.all(np.isfinite(rephasing))
assert np.max(np.abs(esa)) > 1e-6 * reference_amplitude
assert reconstruction_residual < 1e-10

peak_index = np.unravel_index(np.argmax(np.abs(rephasing)), rephasing.shape)
print(f"Pathway reconstruction residual: {reconstruction_residual:.3e}")
print(f"Maximum ESA amplitude: {np.max(np.abs(esa)):.6g}")
print("Strongest total rephasing coordinate:")
print(f"  omega_1q   = {omega_1q[peak_index[0]]:.4f} meV")
print(f"  omega_emit = {omega_emit[peak_index[1]]:.4f} meV")

In [ ]:
plotter = SpectroscopyPlotter(detection_phase=0.0)

pathway_plot = plotter.plot_spectrum_result(
    result_rephasing,
    params={
        "source": "pathways",
        "names": ["GSB", "SE", "ESA"],
        "view": "abs",
        "normalization": "global",
        "labels": (
            r"Emission energy $\omega_{\mathrm{emit}}$ (meV)",
            r"Excitation energy $\omega_{1Q}$ (meV)",
        ),
        "title": r"Pathway-resolved THz $\chi^{(3)}$ rephasing response",
        "diagonals": "auto",
        "style": {"abs_cmap": "magma", "levels": 30, "contour_lines": False},
    },
)

total_plot = plotter.plot_spectrum_result(
    result_rephasing,
    params={
        "source": "components",
        "names": ["rephasing"],
        "view": "all",
        "normalization": "row",
        "labels": (
            r"Emission energy $\omega_{\mathrm{emit}}$ (meV)",
            r"Excitation energy $\omega_{1Q}$ (meV)",
        ),
        "title": r"Generic THz spin-chain $\chi^{(3)}$ rephasing spectrum",
        "diagonals": "auto",
        "style": {"cmap": "RdYlBu_r", "abs_cmap": "magma", "levels": 30, "contour_lines": False},
    },
)

## 6. Dense--sparse scaling

For a chain truncated at two excitations,

$$D(L)=1+L+\binom{L}{2}.$$

The dense backend stores Liouville matrices with $D^4$ complex entries. The sparse backend does not materialize these matrices, although a mixed-state or frequency-domain calculation still carries density vectors with $D^2$ entries. We time backend construction and one linear-response point for $L=4,\ldots,8$. Memory curves are structural estimates for one dense Liouville matrix and one sparse density vector; caches and temporary solver workspaces require additional memory.

In [ ]:
def benchmark_backend(benchmark_model, backend_name):
    options = (
        {"cache_resolvents": False}
        if backend_name == "dense_liouville"
        else {"krylov_tolerance": 1e-11}
    )
    solver = SpectroscopySolver(backend=backend_name, eta=eta, **options)
    start = perf_counter()
    solver.feed_model(benchmark_model)
    build_time = perf_counter() - start

    start = perf_counter()
    value = solver.calc_pathway(
        linear_pathway, linear_protocol, {"omega": bright_k0_energy}
    ).value
    solve_time = perf_counter() - start
    return build_time, solve_time, value


benchmark_rows = []
for length in range(4, 9):
    benchmark_model, _, _, _ = make_spin_chain_model(
        length, J_xy, J_z, g_factor, static_field
    )
    dimension = sum(benchmark_model.dimension(s) for s in benchmark_model.sectors())
    dense_build, dense_solve, dense_value = benchmark_backend(
        benchmark_model, "dense_liouville"
    )
    sparse_build, sparse_solve, sparse_value = benchmark_backend(
        benchmark_model, "sparse_sector"
    )
    assert np.allclose(sparse_value, dense_value, rtol=1e-8, atol=1e-9)
    benchmark_rows.append(
        {
            "L": length,
            "D": dimension,
            "dense_build": dense_build,
            "dense_solve": dense_solve,
            "sparse_build": sparse_build,
            "sparse_solve": sparse_solve,
            "residual": abs(sparse_value - dense_value),
        }
    )

print(" L    D   dense build  sparse build  dense solve  sparse solve  residual")
for row in benchmark_rows:
    print(
        f"{row['L']:2d}  {row['D']:3d}   {row['dense_build']:10.4f}  "
        f"{row['sparse_build']:11.4f}  {row['dense_solve']:10.4f}  "
        f"{row['sparse_solve']:11.4f}  {row['residual']:.2e}"
    )

In [ ]:
timed_lengths = np.asarray([row["L"] for row in benchmark_rows])
dense_total_times = np.asarray(
    [row["dense_build"] + row["dense_solve"] for row in benchmark_rows]
)
sparse_total_times = np.asarray(
    [row["sparse_build"] + row["sparse_solve"] for row in benchmark_rows]
)

memory_lengths = np.arange(4, 13)
memory_dimensions = 1 + memory_lengths + memory_lengths * (memory_lengths - 1) // 2
complex_bytes = np.dtype(np.complex128).itemsize
dense_matrix_mib = complex_bytes * memory_dimensions**4 / 2**20
sparse_vector_mib = complex_bytes * memory_dimensions**2 / 2**20

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0), constrained_layout=True)
axes[0].semilogy(timed_lengths, dense_total_times, "o-", label="dense")
axes[0].semilogy(timed_lengths, sparse_total_times, "o-", label="sparse")
axes[0].set_xlabel("Number of sites $L$")
axes[0].set_ylabel("Build + one-point time (s)")
axes[0].set_title("Measured runtime")
axes[0].legend()

axes[1].semilogy(memory_lengths, dense_matrix_mib, "o-", label=r"dense matrix $D^4$")
axes[1].semilogy(memory_lengths, sparse_vector_mib, "o-", label=r"sparse vector $D^2$")
axes[1].set_xlabel("Number of sites $L$")
axes[1].set_ylabel("Structural memory (MiB)")
axes[1].set_title("Dominant stored object")
axes[1].legend()

## 7. What this example establishes

1. The model is a generic XXZ spin-$1/2$ chain in a longitudinal field, with magnetic parameters in meV and tesla.
2. `ExcitationSectorModel` represents the unequal zero-, one-, and two-magnon manifolds and transforms rectangular magnetic-dipole operators between them.
3. A spatially uniform far-field THz probe selects the bright $k=0$ one-magnon state rather than the complete band.
4. The same magnetic operator connects the one-magnon and interacting two-magnon manifolds, producing a finite ESA contribution.
5. Dense and sparse backends reproduce the same response for sizes accessible to both.
6. The dense Liouville representation has a $D^4$ structural memory cost, while the matrix-free sparse calculation avoids materializing that matrix.

The example is physically parametrized but remains generic: its couplings are illustrative and are not fitted to a specific compound. It assumes a zero-temperature polarized initial state, exact magnon-number conservation, and a phenomenological linewidth. It does not include three-magnon pathways, explicit relaxation channels, transverse static fields, Dzyaloshinskii--Moriya coupling, interchain exchange, or the thermodynamic limit.